In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
import matplotlib.pyplot as plt

# Варіант 1 (Київ)
np.random.seed(42)
city = "Київ"
mean_temp = 9.5          # середньорічна температура, °C
amplitude = 14           # сезонна амплітуда, °C
hum_base = 74            # базова вологість, %

rows = []
for month in range(1, 13):
    temp = mean_temp + amplitude * np.cos((month - 7) / 12 * 2 * np.pi) + np.random.normal(0, 1.0)
    humidity = hum_base + np.random.normal(0, 5.0)
    if temp < 8:
        opal_days = 30       # опалювальний період триває весь місяць
    elif temp < 12:
        opal_days = 15       # опалювальний період триває частину місяця
    else:
        opal_days = 0        # опалення не потрібне

    cost = (
        200
        - 9.0 * temp
        + 0.6 * humidity
        + 2.5 * opal_days
        + np.random.normal(0, 15)
    )
    rows.append({
        "місто": city, "місяць": month,
        "температура": round(temp, 1),
        "вологість": round(humidity, 1),
        "опалювальні_дні": opal_days,
        "витрати_на_опалення": round(cost, 1),
    })

heating = pd.DataFrame(rows)
print(heating)

In [ ]:
# Підготовка даних для моделі з двома предикторами (температура, вологість)
X1 = sm.add_constant(heating[["температура", "вологість"]])
y = heating["витрати_на_опалення"]

model1 = sm.OLS(y, X1).fit()
print(model1.summary())

print(f"\nЗвичайний R²: {model1.rsquared:.4f}")
print(f"Скоригований Adjusted R²: {model1.rsquared_adj:.4f}")

In [ ]:
# Обчислення VIF для моделі з двома предикторами
for i, col in enumerate(X1.columns):
    if col == "const":
        continue
    vif_value = variance_inflation_factor(X1.values, i)
    print(f"VIF для '{col}': {vif_value:.2f}")

In [ ]:
# Побудова другої моделі з трьома предикторами (температура, вологість, опалювальні_дні)
X2 = sm.add_constant(heating[["температура", "вологість", "опалювальні_дні"]])
model2 = sm.OLS(y, X2).fit()

print(model2.summary())

print("\nПорівняння моделей:")
print(f"Модель 1 (2 предиктора): R² = {model1.rsquared:.4f}, Adj. R² = {model1.rsquared_adj:.4f}")
print(f"Модель 2 (3 предиктора): R² = {model2.rsquared:.4f}, Adj. R² = {model2.rsquared_adj:.4f}")

# Перевіримо VIF для трипредикторної моделі
print("\nVIF для моделі з 3 предикторами:")
for i, col in enumerate(X2.columns):
    if col == "const":
        continue
    print(f"VIF для '{col}': {variance_inflation_factor(X2.values, i):.2f}")

In [ ]:
fitted = model1.fittedvalues
residuals = model1.resid

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# 1. Графік залишків проти передбачених значень
ax1.scatter(fitted, residuals, color="purple")
ax1.axhline(0, color="red", linestyle="--")
ax1.set_xlabel("Передбачені значення витрат")
ax1.set_ylabel("Залишки")
ax1.set_title("Графік залишків (Гомоскедастичність)")
ax1.grid(True)

# 2. QQ-plot залишків
stats.probplot(residuals, dist="norm", plot=ax2)
ax2.set_title("QQ-plot залишків")
ax2.grid(True)

plt.tight_layout()
plt.show()

# Тест Шапіро-Вілка на нормальність залишків
shapiro_result = stats.shapiro(residuals)
print(f"Тест Шапіро-Вілка: статистика W = {shapiro_result.statistic:.4f}, p-value = {shapiro_result.pvalue:.4f}")

In [ ]:
# Проста регресія (лише температура)
X_simple = sm.add_constant(heating[["температура"]])
model_simple = sm.OLS(y, X_simple).fit()

print(f"Проста регресія (температура): R² = {model_simple.rsquared:.4f}, Adj. R² = {model_simple.rsquared_adj:.4f}")
print(f"Множинна модель (температура + вологість): Adj. R² = {model1.rsquared_adj:.4f}")

### Відповіді до Завдання 6 та Контрольні запитання:

#### Порівняння з простою регресією:
Додавання другого предиктора (вологості) збільшує загальну пояснену частку дисперсії (Adjusted $R^2$ зростає), оскільки вологість є додатковим незалежним джерелом варіації, яке реально закладене в генерацію цільової змінної.

---

### Відповіді на контрольні питання:

1. **Чому `sm.add_constant()` є обов'язковим кроком перед `sm.OLS()`, і що станеться без нього?**
   Функція додає стовпчик одиниць (`const`), який відповідає за вільний член (перетин $b_0$) рівняння регресії. Якщо його пропустити, регресійна пряма/площина проходитиме через початок координат ($b_0 = 0$), що призведе до хибних і зміщених оцінок решта коефіцієнтів.

2. **У чому коефіцієнт множинної регресії відрізняється від коефіцієнта простої регресії з тим самим предиктором?**
   Коефіцієнт множинної регресії оцінює ефект предиктора **за інших рівних умов** (контролюючи вплив інших змінних у моделі), тоді як у простій регресії цей коефіцієнт відображає сумарний ефект (включно з опосередкованим через інші корелюючі змінні).

3. **Чому звичайний $R^2$ не може зменшитися при додаванні будь-якого предиктора?**
   Звичайний $R^2$ базується на сумі квадратів залишків, яка при додаванні нових регресорів математично не може зрости (або залишається незмінною, або зменшується). Тому він не підходить для порівняння моделей різної складності, на відміну від скоригованого Adjusted $R^2$, який уводить штраф за кожен новий предиктор.

4. **Що означає VIF предиктора близько 1 і вище 10, і що з цим робити на практиці?**
   * **VIF $\approx 1$**: повної або слабкої мультиколінеарності немає, змінні незалежні одна від одної.
   * **VIF $> 10$**: критична мультиколінеарність, стабільність коефіцієнтів руйнується через сильну кореляцію між предикторами.
   * **Що робити**: вилучити один із колінеарних предикторів, об'єднати їх у комбіновану змінну або застосувати методи регуляризації (наприклад, Ridge-регресію).